<a href="https://colab.research.google.com/github/Yike-Ding/structural_bioinformatics/blob/main/03_molecular_docking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**03 Molecular Docking**

download assigned pdb code from github respository

In [1]:
import os
import urllib.request


github_pdb_code_url = "https://github.com/Yike-Ding/structural_bioinformatics/blob/main/data/3AFA_H3.1pdb_code"
filename = "3AFA_H3.1pdb_code"

if not os.path.exists(filename):
  print(f"Downloading {filename} from GitHub...")
  urllib.request.urlretrieve(github_pdb_code_url, filename)
  print("Download completed!")
else:
  print(f"{filename} already exists.")

Download completed!


In [2]:
GITHUB_RAW_PDB_URL = "https://raw.githubusercontent.com/Yike-Ding/structural_bioinformatics/main/data/3AFA_H3.1.pdb"
pdb_path = "3AFA_H3.1.pdb"

if not os.path.exists(pdb_path):
  print(f"Downloading {pdb_path} from GitHub...")
  urllib.request.urlretrieve(GITHUB_RAW_PDB_URL, pdb_path)
  print("Download completed!")
else:
  print(f"{pdb_path} already exists.")

Download completed!


convert .pdb file to .pdbqt format for furture docking

In [3]:
%pip install openbabel
%pip install RDKit

from openbabel import pybel

def pdb_to_pdbqt(pdb_path, pdbqt_path, pH=7.4):
    """
    Convert a PDB file to a PDBQT file needed by docking programs of the AutoDock family.

    Parameters
    ----------
    pdb_path: str or pathlib.Path
        Path to input PDB file.
    pdbqt_path: str or pathlib.path
        Path to output PDBQT file.
    pH: float
        Protonation at given pH.
    """
    molecule = list(pybel.readfile("pdb", str(pdb_path)))[0]
    # add hydrogens at given pH
    molecule.OBMol.CorrectForPH(pH)
    molecule.addh()
    # add partial charges to each atom
    for atom in molecule.atoms:
        atom.OBAtom.GetPartialCharge()
    molecule.write("pdbqt", str(pdbqt_path), overwrite=True)
    return


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 21.9 MB/s eta 0:00:00


In [4]:
pdbqt_path = "3AFA_H3.1.pdbqt"
pdb_to_pdbqt(pdb_path, pdbqt_path, pH=7.4)

Due to the fact that there has not published histone H3.1 ligands, several common small molecules are used for docking experiment.

In [5]:

smiles_list = {
    "aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "ibuprofen": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
}

for name, smi in smiles_list.items():
    mol = pybel.readstring("smi", smi)
    mol.make3D()
    mol.write("pdbqt", f"{name}.pdbqt", overwrite=True)
    print(f"Test small molecule: {name}.pdbqt")

Test small molecule: aspirin.pdbqt
Test small molecule: caffeine.pdbqt
Test small molecule: ibuprofen.pdbqt


Three potential pockets are listed from 02 binding site identification